In [7]:
import timeit, tracemalloc
import numpy as np
import pylops
import warnings
import numpy.linalg as linalg
import sklearn.linear_model as linear_model

In [8]:
def bench(f, *args, number=1, repeat=10):
    """
    Benchmarks function `f(*args)`:
    - number: calls per repeat
    - repeat: number of repeats
    Returns median runtime per call and peak memory usage.
    """
    tracemalloc.start()

    # Run `repeat` times, each running `number` calls
    timings = timeit.repeat(lambda: f(*args), repeat=repeat, number=number)

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    # Median runtime per call
    median_time = np.median(timings) / number
    print(f"Median time per call: {median_time:.6e} s")
    print(f"Peak memory usage: {peak / 1024:.1f} KB")
    return median_time, peak

In [11]:
def update_c(
    F,
    latent_dyn,
    params_update_c={
        "update_c_type": "fista",
        "reg_term": 0.01,
        "smooth_term": 0.01,
        "to_norm_fx": False,
    },
    clear_dyn=[],
    direction="c2n",
    other_params={"warm_start": False},
    random_state=0,
    skip_error=False,
    cofficients=[],
):
    """
    The function comes to update the coefficients of the sub-dynamics, {c_i}, by solving the inverse or solving lasso.
    Inputs:
        F               = list of sub-dynamics. Should be a list of k X k arrays.
        latent_dyn      = latent_dynamics (dynamics dimensions X time)
        params_update_c = dictionary with keys:
            update_c_type  = options:
                 - 'inv' (least squares)
                 - 'lasso' (sklearn lasso)
                 - 'fista' (https://pylops.readthedocs.io/en/latest/api/generated/pylops.optimization.sparsity.FISTA.html)
                 - 'omp' (https://pylops.readthedocs.io/en/latest/gallery/plot_ista.html#sphx-glr-gallery-plot-ista-py)
                 - 'ista' (https://pylops.readthedocs.io/en/latest/api/generated/pylops.optimization.sparsity.ISTA.html)
                 - 'IRLS' (https://pylops.readthedocs.io/en/latest/api/generated/pylops.optimization.sparsity.IRLS.html)
                 - 'spgl1' (https://pylops.readthedocs.io/en/latest/api/generated/pylops.optimization.sparsity.SPGL1.html)


                 - . Refers to the way the coefficients should be claculated (inv -> no l1 regularization)
            reg_term       = scalar between 0 to 1, describe the reg. term on the cofficients
            smooth_term    = scalar between 0 to 1, describe the smooth term on the cofficients (c_t - c_(t-1))
        direction      = can be c2n (clean to noise) OR  n2c (noise to clean)
        other_params   = additional parameters for the lasso solver (optional)
        random_state   = random state for reproducability (optional)
        skip_error     = whether to skip an error when solving the inverse for c (optional)
        cofficients    = needed only if smooth_term > 0. This is the reference coefficients matrix to apply the constraint (c_hat_t - c_(t-1)) on.

    Outputs:
        coefficients matrix (k X T), type = np.array

    example:
    coeffs = update_c(np.random.rand(2,2), np.random.rand(2,15),{})
    """

    if isinstance(latent_dyn, list):
        if len(latent_dyn) == 1:
            several_dyns = False
        else:
            several_dyns = True
    else:
        several_dyns = False
    if several_dyns:
        n_times = latent_dyn[0].shape[1] - 1
    else:
        n_times = latent_dyn.shape[1] - 1

    params_update_c = {
        **{"update_c_type": "inv", "smooth_term": 0, "reg_term": 0},
        **params_update_c,
    }
    if len(clear_dyn) == 0:
        clear_dyn = latent_dyn
    if direction == "n2c":
        latent_dyn, clear_dyn = clear_dyn, latent_dyn
    if isinstance(F, np.ndarray):
        F = [F]
    coeffs_list = []

    for time_point in np.arange(n_times):
        if not several_dyns:
            cur_dyn = clear_dyn[:, time_point]
            next_dyn = latent_dyn[:, time_point + 1]
            total_next_dyn = next_dyn
            f_x_mat = []
            for f_i in F:
                f_x_mat.append(f_i @ cur_dyn)
            stacked_fx = np.vstack(f_x_mat).T
            stacked_fx[stacked_fx > 10**8] = 10**8
        else:
            total_next_dyn = []
            for dyn_num in range(len(latent_dyn)):
                cur_dyn = clear_dyn[dyn_num][:, time_point]
                next_dyn = latent_dyn[dyn_num][:, time_point + 1]
                total_next_dyn.extend(next_dyn.flatten().tolist())
                f_x_mat = []
                for f_num, f_i in enumerate(F):
                    f_x_mat.append(f_i @ cur_dyn)
                if dyn_num == 0:
                    stacked_fx = np.vstack(f_x_mat).T
                else:
                    stacked_fx = np.vstack([stacked_fx, np.vstack(f_x_mat).T])
                stacked_fx[stacked_fx > 10**8] = 10**8

            total_next_dyn = np.reshape(np.array(total_next_dyn), (-1, 1))
        if len(F) == 1:
            stacked_fx = np.reshape(stacked_fx, [-1, 1])
        if params_update_c["smooth_term"] > 0 and time_point > 0:
            if len(cofficients) == 0:
                warnings.warn(
                    "Warning: you called the smoothing option without defining coefficients"
                )
        if (
            params_update_c["smooth_term"] > 0
            and time_point > 0
            and len(cofficients) > 0
        ):
            c_former = cofficients[:, time_point - 1].reshape((-1, 1))
            total_next_dyn_full = np.hstack(
                [total_next_dyn, np.sqrt(params_update_c["smooth_term"]) * c_former]
            )
            stacked_fx_full = np.hstack(
                [
                    stacked_fx,
                    np.sqrt(params_update_c["smooth_term"]) * np.eye(len(stacked_fx)),
                ]
            )
        else:
            total_next_dyn_full = total_next_dyn
            stacked_fx_full = stacked_fx

        if params_update_c["update_c_type"] == "inv" or (
            params_update_c["reg_term"] == 0 and params_update_c["smooth_term"] == 0
        ):
            try:
                coeffs = linalg.pinv(stacked_fx_full) @ total_next_dyn_full.reshape(
                    (-1, 1)
                )
            except:
                if not skip_error:
                    raise NameError(
                        "A problem in taking the inverse of fx when looking for the model coefficients"
                    )
                else:
                    return np.nan * np.ones((len(F), latent_dyn.shape[1]))
        elif params_update_c["update_c_type"] == "lasso":
            clf = linear_model.Lasso(
                alpha=params_update_c["reg_term"],
                random_state=random_state,
                **other_params,
            )
            clf.fit(stacked_fx_full, total_next_dyn_full.T)
            coeffs = np.array(clf.coef_)

        elif params_update_c["update_c_type"].lower() == "fista":
            Aop = pylops.MatrixMult(stacked_fx_full)
            # print('fista')
            if "threshkind" not in params_update_c:
                params_update_c["threshkind"] = "soft"

            coeffs = pylops.optimization.sparsity.FISTA(
                Aop,
                total_next_dyn_full.flatten(),
                niter=params_update_c["num_iters"],
                eps=params_update_c["reg_term"],
                threshkind=params_update_c.get("threshkind"),
            )[0]

        elif params_update_c["update_c_type"].lower() == "ista":
            # print('ista')

            if "threshkind" not in params_update_c:
                params_update_c["threshkind"] = "soft"
            Aop = pylops.MatrixMult(stacked_fx_full)
            coeffs = pylops.optimization.sparsity.ISTA(
                Aop,
                total_next_dyn_full.flatten(),
                niter=params_update_c["num_iters"],
                eps=params_update_c["reg_term"],
                threshkind=params_update_c.get("threshkind"),
            )[0]

        elif params_update_c["update_c_type"].lower() == "omp":
            # print('omp')
            Aop = pylops.MatrixMult(stacked_fx_full)
            coeffs = pylops.optimization.sparsity.OMP(
                Aop,
                total_next_dyn_full.flatten(),
                niter_outer=params_update_c["num_iters"],
                sigma=params_update_c["reg_term"],
            )[0]

        elif params_update_c["update_c_type"].lower() == "spgl1":
            # print('spgl1')
            Aop = pylops.MatrixMult(stacked_fx_full)
            coeffs = pylops.optimization.sparsity.SPGL1(
                Aop,
                total_next_dyn_full.flatten(),
                iter_lim=params_update_c["num_iters"],
                tau=params_update_c["reg_term"],
            )[0]

        elif params_update_c["update_c_type"].lower() == "irls":
            # print('irls')
            Aop = pylops.MatrixMult(stacked_fx_full)

            coeffs = pylops.optimization.sparsity.IRLS(
                Aop,
                total_next_dyn_full.flatten(),
                nouter=50,
                espI=params_update_c["reg_term"],
            )[0]

        else:
            raise NameError("Unknown update c type")
        coeffs_list.append(coeffs.flatten())
    coeffs_final = np.vstack(coeffs_list)

    return coeffs_final.T

In [17]:
num_latents = 30
num_timepoints = 10000
num_motifs = 30

X = np.random.randn(num_latents, num_timepoints)
c = np.random.randn(num_motifs, num_timepoints - 1)
F = [np.random.randn(num_latents, num_latents) for motif in range(num_motifs)]

coeffs = update_c(F, X, {"smooth_term": 0, "reg_term": 0})

# bench(update_c, F, X, {"smooth_term": 0.01, "reg_term": 0.01}, c, number=1, repeat=5)


In [7]:
c.shape

(3, 999)